In [5]:
import json, os
from collections import Counter, defaultdict

projects = ['AAH','BEAM','CB','FH','JBIDE','KEYCLOAK','KOGITO','PROJQUAY']
# Fixed path to match your directory structure
gt_dir = '../ground_truth_2'

all_data = {}
grand_totals = Counter()
grand_combos = Counter()
grand_split_totals = {'train': Counter(), 'val': Counter(), 'test': Counter()}

for proj in projects:
    # Load metadata for split distributions
    meta_path = os.path.join(gt_dir, proj, 'metadata.json')
    with open(meta_path) as f:
        meta = json.load(f)
    
    # Load trace_links for detailed type combos
    links_path = os.path.join(gt_dir, proj, 'trace_links.json')
    with open(links_path) as f:
        links = json.load(f)
    
    # Count specific source_type -> target_type combos
    combos = Counter()
    ltype_counts = Counter()
    for link in links:
        st = link.get('source_type','?')
        tt = link.get('target_type','?')
        lt = link.get('ltype','?')
        combos[f'{st} -> {tt}'] += 1
        ltype_counts[lt] += 1
        grand_combos[f'{st} -> {tt}'] += 1
        grand_totals[lt] += 1
    
    # Load split links for per-split breakdown
    splits_dir = os.path.join(gt_dir, proj, 'splits')
    split_counts = {}
    for split in ['train','val','test']:
        sp = os.path.join(splits_dir, f'{split}_links.json')
        if os.path.exists(sp):
            with open(sp) as f:
                sl = json.load(f)
            sc = Counter(l.get('ltype','?') for l in sl)
            split_counts[split] = {'refinement': sc.get('refinement',0), 'subtask': sc.get('subtask',0), 'total': len(sl)}
            grand_split_totals[split]['refinement'] += sc.get('refinement',0)
            grand_split_totals[split]['subtask'] += sc.get('subtask',0)
            grand_split_totals[split]['total'] += len(sl)
    
    all_data[proj] = {
        'total_links': len(links),
        'total_reqs': meta['num_requirements'],
        'ltype_counts': dict(ltype_counts),
        'combos': dict(combos.most_common()),
        'splits': split_counts,
    }

# Print results
print('='*90)
print('DETAILED CLASS DISTRIBUTION STUDY — Ground Truth Dataset')
print('='*90)

print(f'\n{"Project":<10} {"Reqs":>6} {"Links":>6} {"Refinement":>12} {"Subtask":>10} {"Ref%":>6} {"Sub%":>6}')
print('-'*60)
for proj in projects:
    d = all_data[proj]
    ref = d['ltype_counts'].get('refinement',0)
    sub = d['ltype_counts'].get('subtask',0)
    tot = d['total_links']
    ref_pct = ref/tot*100 if tot else 0
    sub_pct = sub/tot*100 if tot else 0
    print(f'{proj:<10} {d["total_reqs"]:>6} {tot:>6} {ref:>12} {sub:>10} {ref_pct:>5.1f}% {sub_pct:>5.1f}%')

ref_t = grand_totals['refinement']
sub_t = grand_totals['subtask']
all_t = ref_t + sub_t
print('-'*60)
print(f'{"TOTAL":<10} {"":>6} {all_t:>6} {ref_t:>12} {sub_t:>10} {ref_t/all_t*100:>5.1f}% {sub_t/all_t*100:>5.1f}%')

print(f'\n\n{"="*90}')
print('SPECIFIC ISSUE TYPE COMBINATIONS (all projects)')
print('='*90)
for combo, count in grand_combos.most_common():
    pct = count/all_t*100
    print(f'  {combo:<30} {count:>6} ({pct:>5.1f}%)')

print(f'\n\n{"="*90}')
print('PER-PROJECT ISSUE TYPE COMBINATIONS')
print('='*90)
for proj in projects:
    d = all_data[proj]
    print(f'\n  {proj} ({d["total_links"]} links):')
    for combo, count in sorted(d['combos'].items(), key=lambda x: -x[1]):
        pct = count/d['total_links']*100
        print(f'    {combo:<30} {count:>5} ({pct:>5.1f}%)')

print(f'\n\n{"="*90}')
print('SPLIT-LEVEL DISTRIBUTION (train/val/test)')
print('='*90)
print(f'\n{"Project":<10} {"Split":>6} {"Total":>6} {"Ref":>6} {"Sub":>6} {"Ref%":>6} {"Sub%":>6}')
print('-'*50)
for proj in projects:
    d = all_data[proj]
    for split in ['train','val','test']:
        sc = d['splits'].get(split,{})
        tot = sc.get('total',0)
        ref = sc.get('refinement',0)
        sub = sc.get('subtask',0)
        ref_pct = ref/tot*100 if tot else 0
        sub_pct = sub/tot*100 if tot else 0
        print(f'{proj:<10} {split:>6} {tot:>6} {ref:>6} {sub:>6} {ref_pct:>5.1f}% {sub_pct:>5.1f}%')
    print()

print('-'*50)
for split in ['train','val','test']:
    sc = grand_split_totals[split]
    tot = sc['total']
    ref = sc['refinement']
    sub = sc['subtask']
    print(f'{"TOTAL":<10} {split:>6} {tot:>6} {ref:>6} {sub:>6} {ref/tot*100:>5.1f}% {sub/tot*100:>5.1f}%')

DETAILED CLASS DISTRIBUTION STUDY — Ground Truth Dataset

Project      Reqs  Links   Refinement    Subtask   Ref%   Sub%
------------------------------------------------------------
AAH           468    412          370         42  89.8%  10.2%
BEAM         1271   1107            0       1107   0.0% 100.0%
CB           1749   1622            0       1622   0.0% 100.0%
FH           1311   1151          991        160  86.1%  13.9%
JBIDE        3363   3080          208       2872   6.8%  93.2%
KEYCLOAK     2475   2274         1586        688  69.7%  30.3%
KOGITO       1880   1658         1538        120  92.8%   7.2%
PROJQUAY      474    412          397         15  96.4%   3.6%
------------------------------------------------------------
TOTAL              11716         5090       6626  43.4%  56.6%


SPECIFIC ISSUE TYPE COMBINATIONS (all projects)
  Task -> Sub-task                 4852 ( 41.4%)
  Epic -> Task                     3822 ( 32.6%)
  Improvement -> Sub-task          1365 ( 

In [9]:
"""
Stratified Evaluation by Link Type — Frank's Request
=====================================================
Splits test results into Refinement (Epic→Standard) vs Subtask (Standard→Child)
and computes P, R, F1, F2 independently for each category.
Evaluates: LoRA V6, RAG-D, Combined, Zero-Shot
Usage (on H100):
  python stratified_eval.py
"""
import json
import os
from collections import defaultdict, Counter
# ==================== CONFIG ====================
DATA_DIR = "../ground_truth_2"
PROJECTS = ["AAH", "BEAM", "CB", "FH", "JBIDE", "KEYCLOAK", "KOGITO", "PROJQUAY"]
# Prediction paths for each method
METHODS = {
    "LoRA_V6": "../ground_truth_2/LORA_RERUN_2026_05/V6_MLP/PREDICTIONS",
    "RAG_D":   "../ground_truth_2/RAG_RERUN_2026_05/RAG_D/PREDICTIONS",
    "Combined": "../ground_truth_2/COMBINED_RERUN_2026_05/V6_MLP_RAG_D/PREDICTIONS",
    "Zero_Shot": "../ground_truth_2/RESULTS/ZERO_SHOT_H100",
}
# Issue type classification (matches construct_ground_truth_v2.py exactly)
def get_level(itype):
    if itype == 'Epic':
        return 'parent'
    elif itype in ['Story', 'Task', 'Feature', 'Enhancement', 'Bug', 'Improvement']:
        return 'standard'
    elif itype in ['Sub-task', 'Sub-Task']:
        return 'child'
    return None
def get_link_category(src_level, tgt_level):
    if src_level == 'parent' and tgt_level == 'standard':
        return 'refinement'
    elif src_level == 'standard' and tgt_level == 'child':
        return 'subtask'
    return 'other'
def compute_metrics(predictions, labels):
    tp = sum(1 for p, l in zip(predictions, labels) if p == 1 and l == 1)
    fp = sum(1 for p, l in zip(predictions, labels) if p == 1 and l == 0)
    fn = sum(1 for p, l in zip(predictions, labels) if p == 0 and l == 1)
    tn = sum(1 for p, l in zip(predictions, labels) if p == 0 and l == 0)
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    f2 = 5 * precision * recall / (4 * precision + recall) if (4 * precision + recall) else 0.0
    return {
        "P": round(precision, 4), "R": round(recall, 4),
        "F1": round(f1, 4), "F2": round(f2, 4),
        "n": tp + fp + fn + tn,
        "n_pos": tp + fn, "n_neg": fp + tn,
        "tp": tp, "fp": fp, "fn": fn, "tn": tn
    }
def main():
    # Step 1: Build type maps for all projects
    print("=" * 95)
    print("STRATIFIED EVALUATION BY LINK TYPE")
    print("=" * 95)
    type_maps = {}
    for proj in PROJECTS:
        req_path = os.path.join(DATA_DIR, proj, "requirements.json")
        with open(req_path) as f:
            reqs = json.load(f)
        tmap = {}
        for r in reqs:
            itype = (r.get('issue_type') or r.get('type') or r.get('itype') or '').strip()
            tmap[r['id']] = itype
        type_maps[proj] = tmap
    # Step 2: Evaluate each method
    all_results = {}
    for method_name, pred_root in METHODS.items():
        if not os.path.exists(pred_root):
            print(f"\n  [SKIP] {method_name} — path not found: {pred_root}")
            continue
        print(f"\n{'─' * 95}")
        print(f"  METHOD: {method_name}")
        print(f"{'─' * 95}")
        # Accumulate per-category across all projects
        global_buckets = defaultdict(lambda: {"predictions": [], "labels": []})
        per_project = defaultdict(lambda: defaultdict(lambda: {"predictions": [], "labels": []}))
        combo_counts = defaultdict(lambda: Counter())
        for proj in PROJECTS:
            pred_file = os.path.join(pred_root, f"{proj}_predictions.json")
            if not os.path.exists(pred_file):
                print(f"    {proj}: predictions not found, skipping")
                continue
            with open(pred_file) as f:
                preds = json.load(f)
            tmap = type_maps[proj]
            for entry in preds:
                src_id = entry.get("source_id")
                tgt_id = entry.get("target_id")
                label = entry.get("label")
                pred = entry.get("prediction")
                # Conservative scoring: parse failures = predict 0
                if pred is None:
                    pred = 0
                src_type = tmap.get(src_id, "unknown")
                tgt_type = tmap.get(tgt_id, "unknown")
                src_level = get_level(src_type)
                tgt_level = get_level(tgt_type)
                if src_level is None or tgt_level is None:
                    category = "unknown"
                else:
                    category = get_link_category(src_level, tgt_level)
                # Track specific combos for positive pairs
                if label == 1:
                    combo_counts[proj][f"{src_type} -> {tgt_type}"] += 1
                global_buckets[category]["predictions"].append(pred)
                global_buckets[category]["labels"].append(label)
                per_project[proj][category]["predictions"].append(pred)
                per_project[proj][category]["labels"].append(label)
        # Print per-project breakdown
        print(f"\n  {'Project':<12} {'Category':<14} {'n':>6} {'pos':>5} {'neg':>5} "
              f"{'P':>7} {'R':>7} {'F1':>7} {'F2':>7}")
        print("  " + "-" * 85)
        for proj in PROJECTS:
            proj_data = per_project.get(proj, {})
            if not proj_data:
                continue
            for cat in ["refinement", "subtask", "other", "unknown"]:
                if cat not in proj_data:
                    continue
                bucket = proj_data[cat]
                if not bucket["predictions"]:
                    continue
                m = compute_metrics(bucket["predictions"], bucket["labels"])
                print(f"  {proj:<12} {cat:<14} {m['n']:>6} {m['n_pos']:>5} {m['n_neg']:>5} "
                      f"{m['P']:>7.4f} {m['R']:>7.4f} {m['F1']:>7.4f} {m['F2']:>7.4f}")
            print()
        # Print global macro (per-project average within each category)
        print(f"\n  {'=' * 85}")
        print(f"  MACRO AVERAGES (per-project average within each category)")
        print(f"  {'=' * 85}")
        print(f"  {'Category':<14} {'Projects':>8} {'Total n':>8} "
              f"{'Macro P':>8} {'Macro R':>8} {'Macro F1':>9} {'Macro F2':>9}")
        print("  " + "-" * 70)
        for cat in ["refinement", "subtask"]:
            cat_projects = []
            total_n = 0
            for proj in PROJECTS:
                if proj in per_project and cat in per_project[proj]:
                    bucket = per_project[proj][cat]
                    if bucket["predictions"]:
                        m = compute_metrics(bucket["predictions"], bucket["labels"])
                        if m["n_pos"] > 0:  # Only include projects that have positive links of this type
                            cat_projects.append(m)
                            total_n += m["n"]
            if cat_projects:
                macro_p  = sum(m["P"] for m in cat_projects) / len(cat_projects)
                macro_r  = sum(m["R"] for m in cat_projects) / len(cat_projects)
                macro_f1 = sum(m["F1"] for m in cat_projects) / len(cat_projects)
                macro_f2 = sum(m["F2"] for m in cat_projects) / len(cat_projects)
                print(f"  {cat:<14} {len(cat_projects):>8} {total_n:>8} "
                      f"{macro_p:>8.4f} {macro_r:>8.4f} {macro_f1:>9.4f} {macro_f2:>9.4f}")
        all_results[method_name] = {cat: dict(global_buckets[cat]) for cat in global_buckets}
    # Final comparison table
    print(f"\n\n{'=' * 95}")
    print("FINAL COMPARISON — Refinement vs Subtask by Method (Macro F2)")
    print(f"{'=' * 95}")
    print(f"\n  {'Method':<14} {'Ref F2':>8} {'Sub F2':>8} {'Δ (Sub-Ref)':>12} {'Observation'}")
    print("  " + "-" * 70)
    for method_name in METHODS:
        if method_name not in all_results:
            continue
        # Recompute macro per category
        for cat in ["refinement", "subtask"]:
            pass  # Already printed above
    print("\n  (See per-method sections above for detailed numbers)")
    print(f"\n{'=' * 95}")
if __name__ == "__main__":
    main()


STRATIFIED EVALUATION BY LINK TYPE

───────────────────────────────────────────────────────────────────────────────────────────────
  METHOD: LoRA_V6
───────────────────────────────────────────────────────────────────────────────────────────────

  Project      Category            n   pos   neg       P       R      F1      F2
  -------------------------------------------------------------------------------------
  AAH          refinement        500   125   375  0.4577  0.5200  0.4869  0.5062
  AAH          subtask            48    12    36  1.0000  0.8333  0.9091  0.8621

  BEAM         subtask          1024   256   768  0.4252  0.8438  0.5654  0.7050

  CB           subtask          1416   354  1062  0.6959  0.9633  0.8081  0.8945

  FH           refinement       1172   293   879  0.7525  0.7679  0.7601  0.7648
  FH           subtask           160    40   120  0.4697  0.7750  0.5849  0.6858

  JBIDE        refinement        340    85   255  0.3039  0.6471  0.4135  0.5278
  JBIDE      